# **ADNI Multi-Table Pipeline**

In [ ]:
# =========================
# ADNI Multi-Table Pipeline
# =========================
# - Loads ALL provided ADNI tables
# - Cleans keys (RID + VISCODE), consolidates PTDEMOG to 1 row/RID, builds APOE features
# - Merges into a single visit-level dataset (anchored on ADNIMERGE)
# - Splits by RID BEFORE any transformations (to avoid leakage)
# - Drops high-missingness columns using TRAIN ONLY (threshold > 0.30; here = 0.45)
# - Imputes, transforms skew, scales, encodes categorical features (fit on TRAIN ONLY)
# - Applies SMOTE on TRAIN ONLY
# - PCA (90% variance) fit on TRAIN ONLY
#
# Outputs:
# - X_train_pca, X_val_pca, X_test_pca
# - y_train_res (SMOTE), y_val, y_test
# - merged_raw.csv, and optional artifacts saved via joblib

In [6]:
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, PowerTransformer
from sklearn.decomposition import PCA

from imblearn.over_sampling import SMOTE

import joblib

In [ ]:
# -------------------------
# 0) Config
# -------------------------
RANDOM_STATE = 42

# Your request: "increase the threshold well past 0.30"
MISSINGNESS_THRESHOLD = 0.45

# Split ratios: 70 / 15 / 15 (train / val / test), but split BY RID (group split)
TEST_VAL_TOTAL = 0.30
VAL_IN_TEMP = 0.50  # 0.30 temp -> 0.15 val + 0.15 test

# File paths (adjust if needed)
PATH_ADNIMERGE = "ADNIMERGE_10Nov2025.csv"
PATH_CDR       = "Dementia_Rating.csv"
PATH_DIAG      = "Diagnositic_Summary.csv"
PATH_COG       = "Cognitive_Scores.csv"
PATH_PTDEMOG   = "PTDEMOG_10Nov2025.csv"
PATH_APOE      = "ApoE_Genotyping.csv"

In [ ]:
# -------------------------
# 1) Helpers
# -------------------------
def _to_str(x):
    if pd.isna(x):
        return np.nan
    return str(x)

def normalize_viscode(series: pd.Series) -> pd.Series:
    """
    Standardize visit code strings for safer joins.
    Keeps baseline-like tokens; lowercases; trims whitespace.
    """
    s = series.astype("string").str.strip().str.lower()
    # common normalizations
    s = s.str.replace(r"\s+", "", regex=True)
    return s

def build_join_viscode(df: pd.DataFrame) -> pd.Series:
    """
    Creates VISCODE_JOIN:
    - Prefer VISCODE if exists and non-null
    - Else fall back to VISCODE2 if exists
    """
    vis = pd.Series([np.nan] * len(df), index=df.index, dtype="string")
    if "VISCODE" in df.columns:
        vis = normalize_viscode(df["VISCODE"])
    if "VISCODE2" in df.columns:
        vis2 = normalize_viscode(df["VISCODE2"])
        vis = vis.fillna(vis2)
    return vis

def safe_to_datetime(df: pd.DataFrame, col: str) -> None:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

def parse_vismonth_from_viscode(viscode: pd.Series) -> pd.Series:
    """
    Extracts numeric month index from common ADNI visit codes.
    Examples:
      m06 -> 6, m12 -> 12, bl/sc -> 0
    Non-m patterns become NaN.
    """
    s = viscode.astype("string").str.lower()
    out = pd.Series([np.nan] * len(s), index=s.index, dtype="float")

    # baseline-ish
    out = out.mask(s.isin(["bl", "sc", "scr", "screen", "screening", "init", "m00"]), 0)

    # mXX
    m = s.str.extract(r"^m(\d+)$")[0]
    out = out.mask(m.notna(), m.astype(float))

    return out

def clean_dx_labels(dx: pd.Series) -> pd.Series:
    """
    Normalize diagnosis labels to a consistent set.
    Keeps: CN, MCI, Dementia
    """
    s = dx.astype("string").str.strip().str.upper()

    mapping = {
        "NORMAL": "CN",
        "NC": "CN",
        "CN": "CN",
        "COGNITIVELY NORMAL": "CN",

        "MCI": "MCI",
        "EMCI": "MCI",
        "LMCI": "MCI",

        "AD": "DEMENTIA",
        "ALZHEIMER'S DISEASE": "DEMENTIA",
        "DEMENTIA": "DEMENTIA",
    }
    s = s.replace(mapping)

    # unify case for final
    s = s.replace({"DEMENTIA": "Dementia", "MCI": "MCI", "CN": "CN"})

    # keep only these three classes (drop everything else to keep task clean)
    s = s.where(s.isin(["CN", "MCI", "Dementia"]), np.nan)
    return s

def mode_label(x: pd.Series):
    x = x.dropna()
    if len(x) == 0:
        return np.nan
    return x.value_counts().idxmax()

def first_non_null(series: pd.Series):
    s = series.dropna()
    return s.iloc[0] if len(s) else np.nan

def consolidate_ptdemog_to_dim_subject(ptdemog: pd.DataFrame) -> pd.DataFrame:
    """
    PTDEMOG has multiple rows per RID (multi-phase). Consolidate to 1 row per RID.
    Strategy:
      - If update_stamp exists: sort DESC (latest), then take first non-null per column
      - Else: take first non-null per column without sorting
    """
    df = ptdemog.copy()

    if "RID" not in df.columns:
        raise ValueError("PTDEMOG must contain RID.")

    # parse update_stamp if present
    if "update_stamp" in df.columns:
        safe_to_datetime(df, "update_stamp")
        df = df.sort_values(["RID", "update_stamp"], ascending=[True, False])
    else:
        df = df.sort_values(["RID"])

    # choose a conservative set of demographic columns if present (keep others too)
    # We'll aggregate ALL columns, but you can restrict if desired.
    agg_dict = {c: first_non_null for c in df.columns if c != "RID"}
    dim = df.groupby("RID", as_index=False).agg(agg_dict)

    return dim

def build_apoe_features(apoe: pd.DataFrame) -> pd.DataFrame:
    """
    Create APOE4_count and APOE4_carrier from GENOTYPE.
    """
    df = apoe.copy()
    if "RID" not in df.columns:
        raise ValueError("ApoE_Genotyping must contain RID.")
    if "GENOTYPE" not in df.columns:
        # If genotype isn't present, still return RID unique rows
        out = df[["RID"]].drop_duplicates().copy()
        out["APOE4_count"] = np.nan
        out["APOE4_carrier"] = np.nan
        return out

    gt = df["GENOTYPE"].astype("string").str.strip()
    # count of '4' alleles in strings like '3/4', '4/4'
    apoe4_count = gt.str.findall("4").apply(lambda lst: len(lst) if isinstance(lst, list) else np.nan)
    df["APOE4_count"] = apoe4_count.astype("float")
    df["APOE4_carrier"] = (df["APOE4_count"] >= 1).astype("float")

    # one row per RID (keep first non-null)
    df = df.sort_values(["RID"])
    out = df.groupby("RID", as_index=False).agg({
        "GENOTYPE": first_non_null,
        "APOE4_count": first_non_null,
        "APOE4_carrier": first_non_null
    })
    return out

def drop_leaky_columns(df: pd.DataFrame, target_col: str = "DX_CLEAN") -> pd.DataFrame:
    """
    Remove columns that directly encode diagnosis labels or are label-derived.
    Keeps the target_col (default DX_CLEAN).
    """
    df = df.copy()

    # Exact leaky columns to drop (but keep target_col)
    leaky_exact = {
        "DIAGNOSIS", "PHC_Diagnosis",
        "DX_bl", "DXCHANGE"
    }

    to_drop = []
    for c in df.columns:
        cup = str(c).upper()

        # keep target, always
        if c == target_col:
            continue

        # drop exact known label-derived cols
        if c in leaky_exact:
            to_drop.append(c)
            continue

        # drop diagnostic-summary DX* flags like DXNORM, DXMCI, DXAD, DXCONFID, etc.
        # but DO NOT drop plain "DX" here (you may want it for audit; you'll drop it later from features anyway)
        if cup.startswith("DX") and c != "DX" and c != target_col:
            # Example matches: DXNORM, DXMCI, DXDEP, DXPARK, DXCONFID...
            to_drop.append(c)

    to_drop = [c for c in to_drop if c in df.columns]
    return df.drop(columns=to_drop, errors="ignore")

def make_onehot_encoder():
    """
    Compatibility for sklearn versions:
      - newer: OneHotEncoder(sparse_output=False)
      - older: OneHotEncoder(sparse=False)
    """
    try:
        return OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    except TypeError:
        return OneHotEncoder(sparse=False, handle_unknown="ignore")

In [ ]:
# -------------------------
# 2) Load all datasets
# -------------------------
adni = pd.read_csv(PATH_ADNIMERGE, low_memory=False)
cdr  = pd.read_csv(PATH_CDR, low_memory=False)
diag = pd.read_csv(PATH_DIAG, low_memory=False)
cog  = pd.read_csv(PATH_COG, low_memory=False)
ptd  = pd.read_csv(PATH_PTDEMOG, low_memory=False)
apoe = pd.read_csv(PATH_APOE, low_memory=False)

# Normalize RID type where possible
for df in [adni, cdr, diag, cog, ptd, apoe]:
    if "RID" in df.columns:
        df["RID"] = pd.to_numeric(df["RID"], errors="coerce").astype("Int64")

# Normalize visit codes into a common join column
for df in [adni, cdr, diag, cog]:
    df["VISCODE_JOIN"] = build_join_viscode(df)

# Parse date columns if present (useful for audit / feature engineering)
for col in ["EXAMDATE", "VISDATE", "update_stamp", "APTESTDT"]:
    safe_to_datetime(adni, col)
    safe_to_datetime(cdr, col)
    safe_to_datetime(diag, col)
    safe_to_datetime(cog, col)
    safe_to_datetime(ptd, col)
    safe_to_datetime(apoe, col)

# Optional QC filtering (CDR has HAS_QC_ERROR)
if "HAS_QC_ERROR" in cdr.columns:
    cdr = cdr[cdr["HAS_QC_ERROR"].fillna(0).astype(int) == 0].copy()

# Ensure uniqueness on (RID, VISCODE_JOIN) for visit-level tables (keep first if duplicates)
def dedup_visit_table(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if "RID" not in df.columns or "VISCODE_JOIN" not in df.columns:
        return df
    df2 = df.copy()
    df2 = df2.sort_values(["RID", "VISCODE_JOIN"])
    df2 = df2.drop_duplicates(subset=["RID", "VISCODE_JOIN"], keep="first")
    return df2

adni = dedup_visit_table(adni, "ADNIMERGE")
cdr  = dedup_visit_table(cdr,  "Dementia_Rating")
diag = dedup_visit_table(diag, "Diagnositic_Summary")
cog  = dedup_visit_table(cog,  "Cognitive_Scores")

# Consolidate PTDEMOG to dim_subject (1 row per RID)
dim_subject = consolidate_ptdemog_to_dim_subject(ptd)

# Build APOE features (1 row per RID)
dim_genetics = build_apoe_features(apoe)

In [ ]:
# -------------------------
# 3) Build merged RAW dataset (anchored on ADNIMERGE visit facts)
# -------------------------
# Normalize target DX
if "DX" not in adni.columns:
    raise ValueError("ADNIMERGE must contain DX as the target label.")
adni["DX_CLEAN"] = clean_dx_labels(adni["DX"])

# Keep only rows with valid target
adni_clean = adni.dropna(subset=["DX_CLEAN", "RID", "VISCODE_JOIN"]).copy()

# Add a useful numeric visit-month feature
adni_clean["VISMONTH"] = parse_vismonth_from_viscode(adni_clean["VISCODE_JOIN"])

# Merge visit-level add-ons (left join so we keep ADNIMERGE backbone)
merged = adni_clean.merge(
    cdr.drop(columns=["VISCODE", "VISCODE2"], errors="ignore"),
    on=["RID", "VISCODE_JOIN"],
    how="left",
    suffixes=("", "_cdr")
)

merged = merged.merge(
    diag.drop(columns=["VISCODE", "VISCODE2"], errors="ignore"),
    on=["RID", "VISCODE_JOIN"],
    how="left",
    suffixes=("", "_diag")
)

merged = merged.merge(
    cog.drop(columns=["VISCODE", "VISCODE2"], errors="ignore"),
    on=["RID", "VISCODE_JOIN"],
    how="left",
    suffixes=("", "_cog")
)

# Merge subject dimensions (RID only)
merged = merged.merge(dim_subject, on="RID", how="left", suffixes=("", "_ptd"))
merged = merged.merge(dim_genetics, on="RID", how="left", suffixes=("", "_apoe"))

# Drop obvious label-leakage columns coming from other tables
merged = drop_leaky_columns(merged, target_col="DX_CLEAN")

# Save raw merged for auditing
merged.to_csv("merged_adni_raw.csv", index=False)

print("Merged raw shape:", merged.shape)
print("Class distribution (DX_CLEAN):")
print(merged["DX_CLEAN"].value_counts())

Merged raw shape: (11458, 260)
Class distribution (DX_CLEAN):
DX_CLEAN
MCI         4989
CN          4020
Dementia    2449
Name: count, dtype: Int64


In [ ]:
# -------------------------
# 4) Split by RID BEFORE transformations (no leakage)
# -------------------------
target = "DX_CLEAN"

# Define X/y, keep RID for grouping then drop from features later
X_all = merged.copy()
y_all = X_all[target].copy()

# Build group-level label for stratification (mode across visits)
group_labels = (
    pd.DataFrame({"RID": X_all["RID"], "y": y_all})
    .dropna()
    .groupby("RID")["y"]
    .agg(mode_label)
    .dropna()
)

rids = group_labels.index.astype("int")
rid_y = group_labels.values

# 1) RID split: train vs temp
rid_train, rid_temp = train_test_split(
    rids,
    test_size=TEST_VAL_TOTAL,
    random_state=RANDOM_STATE,
    stratify=rid_y
)

# 2) temp -> val + test
# Need stratify for temp too
temp_labels = pd.Series(rid_y, index=rids).loc[rid_temp].values
rid_val, rid_test = train_test_split(
    rid_temp,
    test_size=VAL_IN_TEMP,
    random_state=RANDOM_STATE,
    stratify=temp_labels
)

# Now create row-level splits by RID membership
train_mask = X_all["RID"].isin(rid_train)
val_mask   = X_all["RID"].isin(rid_val)
test_mask  = X_all["RID"].isin(rid_test)

X_train = X_all.loc[train_mask].copy()
y_train = y_all.loc[train_mask].copy()

X_val = X_all.loc[val_mask].copy()
y_val = y_all.loc[val_mask].copy()

X_test = X_all.loc[test_mask].copy()
y_test = y_all.loc[test_mask].copy()

print("\nRow-level split sizes:")
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

print("\nRID-level split sizes:")
print("Train RIDs:", len(rid_train), "Val RIDs:", len(rid_val), "Test RIDs:", len(rid_test))


Row-level split sizes:
Train: (8004, 260) Val: (1714, 260) Test: (1740, 260)

RID-level split sizes:
Train RIDs: 1686 Val RIDs: 361 Test RIDs: 362


In [ ]:
# -------------------------
# 5) Drop columns with high missingness (TRAIN ONLY)
#    (Threshold > 0.30; using 0.45 per your request)
# -------------------------
# Remove target + high-leak identifiers from features
IDENTIFIER_COLS = [
    "DX", "DX_CLEAN", "RID", "PTID", "VISCODE", "VISCODE2", "VISCODE_JOIN",
    "EXAMDATE", "VISDATE", "update_stamp", "APTESTDT"
]

X_train_feat = X_train.drop(columns=[c for c in IDENTIFIER_COLS if c in X_train.columns], errors="ignore")
X_val_feat   = X_val.drop(columns=[c for c in IDENTIFIER_COLS if c in X_val.columns], errors="ignore")
X_test_feat  = X_test.drop(columns=[c for c in IDENTIFIER_COLS if c in X_test.columns], errors="ignore")

# Compute missingness on TRAIN ONLY
missing_rates = X_train_feat.isna().mean()
cols_to_drop = missing_rates[missing_rates > MISSINGNESS_THRESHOLD].index.tolist()

X_train_feat = X_train_feat.drop(columns=cols_to_drop)
X_val_feat   = X_val_feat.drop(columns=cols_to_drop, errors="ignore")
X_test_feat  = X_test_feat.drop(columns=cols_to_drop, errors="ignore")

print("\nDropped columns due to missingness >", MISSINGNESS_THRESHOLD, ":", len(cols_to_drop))
print("Feature shapes after dropping:", X_train_feat.shape, X_val_feat.shape, X_test_feat.shape)


Dropped columns due to missingness > 0.45 : 139
Feature shapes after dropping: (8004, 111) (1714, 111) (1740, 111)


In [ ]:
# -------------------------
# 6) Identify numeric vs categorical (after dropping)
# -------------------------
num_cols = X_train_feat.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train_feat.columns.difference(num_cols).tolist()

print("\n# Numeric cols:", len(num_cols), " | # Categorical cols:", len(cat_cols))


# Numeric cols: 88  | # Categorical cols: 23


In [ ]:
# -------------------------
# 7) Impute (fit on TRAIN ONLY)
# -------------------------
# Numeric impute
num_imputer = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(num_imputer.fit_transform(X_train_feat[num_cols]), columns=num_cols, index=X_train_feat.index)
X_val_num   = pd.DataFrame(num_imputer.transform(X_val_feat[num_cols]), columns=num_cols, index=X_val_feat.index)
X_test_num  = pd.DataFrame(num_imputer.transform(X_test_feat[num_cols]), columns=num_cols, index=X_test_feat.index)

# Categorical impute
cat_imputer = SimpleImputer(strategy="most_frequent")
X_train_cat = pd.DataFrame(cat_imputer.fit_transform(X_train_feat[cat_cols]), columns=cat_cols, index=X_train_feat.index)
X_val_cat   = pd.DataFrame(cat_imputer.transform(X_val_feat[cat_cols]), columns=cat_cols, index=X_val_feat.index)
X_test_cat  = pd.DataFrame(cat_imputer.transform(X_test_feat[cat_cols]), columns=cat_cols, index=X_test_feat.index)

print("\nNaNs after imputation:",
      X_train_num.isna().sum().sum() + X_train_cat.isna().sum().sum(),
      X_val_num.isna().sum().sum() + X_val_cat.isna().sum().sum(),
      X_test_num.isna().sum().sum() + X_test_cat.isna().sum().sum()
)


NaNs after imputation: 0 0 0


In [ ]:
# -------------------------
# 8) Skew handling (TRAIN ONLY decisions)
#    - heavy skew: log1p with train-based shift
#    - moderate skew: Yeo-Johnson fit on train
# -------------------------
skews = X_train_num.skew(numeric_only=True)

X_train_tf = X_train_num.copy()
X_val_tf   = X_val_num.copy()
X_test_tf  = X_test_num.copy()

yeojohnson_models = {}
log_shifts = {}

for col in num_cols:
    sk = skews[col]

    if abs(sk) >= 1.0:
        # log transform with safe shifting based on TRAIN
        minv = X_train_tf[col].min()
        shift = (-minv + 1e-6) if minv <= -1 else 0.0
        log_shifts[col] = shift

        X_train_tf[col] = np.log1p(X_train_tf[col] + shift)
        X_val_tf[col]   = np.log1p(X_val_tf[col] + shift)
        X_test_tf[col]  = np.log1p(X_test_tf[col] + shift)

    elif abs(sk) >= 0.5:
        # Yeo-Johnson fit on TRAIN only
        pt = PowerTransformer(method="yeo-johnson")
        pt.fit(X_train_tf[[col]])
        yeojohnson_models[col] = pt

        X_train_tf[col] = pt.transform(X_train_tf[[col]])
        X_val_tf[col]   = pt.transform(X_val_tf[[col]])
        X_test_tf[col]  = pt.transform(X_test_tf[[col]])

In [ ]:
# -------------------------
# 9) Scale numeric (fit on TRAIN ONLY)
# -------------------------
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_tf), columns=num_cols, index=X_train_tf.index)
X_val_scaled   = pd.DataFrame(scaler.transform(X_val_tf), columns=num_cols, index=X_val_tf.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test_tf), columns=num_cols, index=X_test_tf.index)

In [ ]:
# -------------------------
# 10) Encode categorical (fit on TRAIN ONLY)
#     - binary -> OrdinalEncoder
#     - multi-class -> OneHotEncoder(handle_unknown=ignore)
# -------------------------
encoded_train = []
encoded_val   = []
encoded_test  = []

cat_encoders = {}

for col in cat_cols:
    nunique = X_train_cat[col].nunique(dropna=False)

    if nunique <= 2:
        enc = OrdinalEncoder()
        enc.fit(X_train_cat[[col]])
        cat_encoders[col] = ("ordinal", enc)

        encoded_train.append(pd.DataFrame(enc.transform(X_train_cat[[col]]), columns=[col], index=X_train_cat.index))
        encoded_val.append(pd.DataFrame(enc.transform(X_val_cat[[col]]), columns=[col], index=X_val_cat.index))
        encoded_test.append(pd.DataFrame(enc.transform(X_test_cat[[col]]), columns=[col], index=X_test_cat.index))

    else:
        enc = make_onehot_encoder()
        enc.fit(X_train_cat[[col]])
        cat_encoders[col] = ("onehot", enc)

        oh_cols = [f"{col}_{c}" for c in enc.categories_[0]]
        encoded_train.append(pd.DataFrame(enc.transform(X_train_cat[[col]]), columns=oh_cols, index=X_train_cat.index))
        encoded_val.append(pd.DataFrame(enc.transform(X_val_cat[[col]]), columns=oh_cols, index=X_val_cat.index))
        encoded_test.append(pd.DataFrame(enc.transform(X_test_cat[[col]]), columns=oh_cols, index=X_test_cat.index))

X_train_cat_enc = pd.concat(encoded_train, axis=1) if encoded_train else pd.DataFrame(index=X_train_feat.index)
X_val_cat_enc   = pd.concat(encoded_val, axis=1)   if encoded_val else pd.DataFrame(index=X_val_feat.index)
X_test_cat_enc  = pd.concat(encoded_test, axis=1)  if encoded_test else pd.DataFrame(index=X_test_feat.index)

# Combine
X_train_final = pd.concat([X_train_scaled, X_train_cat_enc], axis=1)
X_val_final   = pd.concat([X_val_scaled,   X_val_cat_enc], axis=1)
X_test_final  = pd.concat([X_test_scaled,  X_test_cat_enc], axis=1)

# Align columns
X_train_final, X_val_final = X_train_final.align(X_val_final, join="left", axis=1, fill_value=0)
X_train_final, X_test_final = X_train_final.align(X_test_final, join="left", axis=1, fill_value=0)

print("\nFinal feature matrices:")
print("Train:", X_train_final.shape, "Val:", X_val_final.shape, "Test:", X_test_final.shape)


Final feature matrices:
Train: (8004, 8614) Val: (1714, 8614) Test: (1740, 8614)


In [ ]:
# -------------------------
# 11) SMOTE (TRAIN ONLY)
# -------------------------
print("\nClass balance BEFORE SMOTE:")
print(y_train.value_counts())

sm = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = sm.fit_resample(X_train_final, y_train)

print("\nClass balance AFTER SMOTE:")
print(pd.Series(y_train_res).value_counts())


Class balance BEFORE SMOTE:
DX_CLEAN
MCI         3453
CN          2851
Dementia    1700
Name: count, dtype: Int64

Class balance AFTER SMOTE:
DX_CLEAN
CN          3453
MCI         3453
Dementia    3453
Name: count, dtype: Int64


In [ ]:
# -------------------------
# 12) PCA (fit on TRAIN ONLY)
# -------------------------
pca = PCA(n_components=0.90, svd_solver="full", random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_res)
X_val_pca   = pca.transform(X_val_final)
X_test_pca  = pca.transform(X_test_final)

print("\nPCA shapes:", X_train_pca.shape, X_val_pca.shape, X_test_pca.shape)


PCA shapes: (10359, 56) (1714, 56) (1740, 56)


In [ ]:
# -------------------------
# 13) Save artifacts
# -------------------------
np.save("X_train_pca.npy", X_train_pca)
np.save("X_val_pca.npy",   X_val_pca)
np.save("X_test_pca.npy",  X_test_pca)

joblib.dump({
    "missingness_threshold": MISSINGNESS_THRESHOLD,
    "cols_dropped_missingness": cols_to_drop,
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "num_imputer": num_imputer,
    "cat_imputer": cat_imputer,
    "log_shifts": log_shifts,
    "yeojohnson_models": yeojohnson_models,
    "scaler": scaler,
    "cat_encoders": cat_encoders,
    "pca": pca,
    "final_columns": X_train_final.columns.tolist()
}, "preprocessing_artifacts.joblib")

# Also save labels
y_train_res.to_csv("y_train_resampled.csv", index=False)
y_val.to_csv("y_val.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("\nSaved: merged_adni_raw.csv, X_*_pca.npy, y_*.csv, preprocessing_artifacts.joblib")


Saved: merged_adni_raw.csv, X_*_pca.npy, y_*.csv, preprocessing_artifacts.joblib


In [ ]:
df = pd.read_csv("merged_adni_raw.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11458 entries, 0 to 11457
Columns: 260 entries, RID to APOE4_carrier
dtypes: float64(195), int64(7), object(58)
memory usage: 22.7+ MB
